# Data quality checks

**Do quality checks need more than one machine?**

Read-only passes over the whole table. Worth attention because if the answer is no, these are the
easiest jobs to move: nothing is written, nothing downstream breaks if one fails, and they are
trivial to run both ways and compare.

The data has known defects planted in it, listed in notebook 00, so these checks have something
real to find:

* ~2% null customer IDs and ~1% null amounts
* ~0.5% exact duplicate rows
* ~0.5% negative amounts, which are invalid
* ~1% of rows pointing at customers that do not exist
* inconsistent country text

**Q6** reconciles two tables in both directions.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "06_data_quality", C.MAIN_SIZE)
print(f"Ready. {C.human(C.MAIN_SIZE)} sales rows. "
      f"Both engines have {C.ENGINE_MEMORY_MB} MB and {C.plural(C.ENGINE_THREADS, 'thread')}.")


In [ ]:
SQL_Q1 = """
SELECT count(*) AS n_rows,
  count(*) - count(customer_id)     AS null_customer_id,
  count(*) - count(amount)          AS null_amount,
  count(*) - count(promo_code)      AS null_promo_code,
  count(*) - count(product_id)      AS null_product_id,
  count(*) - count(sale_date)       AS null_sale_date,
  count(*) - count(channel)         AS null_channel,
  count(*) - count(region)          AS null_region,
  count(*) - count(session_ref)     AS null_session_ref
FROM sales
"""

_, out, _ = bench.run(Case("Q1", "Null profile across all columns", "Data quality", sql=SQL_Q1.strip()))
display(out.head())


In [ ]:
SQL_Q2 = """
SELECT count(*) AS duplicated_keys, COALESCE(sum(c),0) AS extra_rows FROM (
  SELECT sale_id, count(*) - 1 AS c FROM sales GROUP BY 1 HAVING count(*) > 1
) t
"""

_, out, _ = bench.run(Case("Q2", "Duplicate detection", "Data quality", sql=SQL_Q2.strip()))
display(out.head())


In [ ]:
SQL_Q3 = """
SELECT
  sum(CASE WHEN amount < 0 THEN 1 ELSE 0 END)                     AS negative_amount,
  sum(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END)                  AS bad_quantity,
  sum(CASE WHEN discount_pct < 0 OR discount_pct > 0.9 THEN 1 ELSE 0 END) AS bad_discount,
  sum(CASE WHEN sale_date < DATE '2024-01-01' THEN 1 ELSE 0 END)  AS date_too_early,
  sum(CASE WHEN currency NOT IN ('GBP','USD','EUR') THEN 1 ELSE 0 END) AS bad_currency
FROM sales
"""

_, out, _ = bench.run(Case("Q3", "Range and validity checks", "Data quality", sql=SQL_Q3.strip()))
display(out.head())


In [ ]:
SQL_Q4 = """
SELECT
  count(*) AS total,
  sum(CASE WHEN c.customer_id IS NULL AND s.customer_id IS NOT NULL THEN 1 ELSE 0 END) AS missing_customer,
  sum(CASE WHEN p.product_id  IS NULL THEN 1 ELSE 0 END)                               AS missing_product
FROM sales s
LEFT JOIN customers c ON c.customer_id = s.customer_id
LEFT JOIN products  p ON p.product_id  = s.product_id
"""

_, out, _ = bench.run(Case("Q4", "Referential integrity", "Data quality", sql=SQL_Q4.strip()))
display(out.head())


In [ ]:
SQL_Q5 = """
SELECT
  count(DISTINCT channel) AS d_channel, count(DISTINCT device) AS d_device,
  count(DISTINCT region) AS d_region, count(DISTINCT currency) AS d_currency,
  count(DISTINCT payment_method) AS d_payment, count(DISTINCT store_id) AS d_store,
  min(amount) AS min_amount, max(amount) AS max_amount,
  round(avg(amount),3) AS avg_amount, round(stddev(amount),3) AS sd_amount,
  min(sale_date) AS min_date, max(sale_date) AS max_date
FROM sales
"""

_, out, _ = bench.run(Case("Q5", "Full column profile", "Data quality", sql=SQL_Q5.strip()))
display(out.head())


In [ ]:
SQL_Q6 = """
SELECT
  (SELECT count(*) FROM (SELECT sale_id FROM sales EXCEPT SELECT sale_id FROM sales_v2) a) AS only_in_v1,
  (SELECT count(*) FROM (SELECT sale_id FROM sales_v2 EXCEPT SELECT sale_id FROM sales) b) AS only_in_v2
"""

_, out, _ = bench.run(Case("Q6", "Reconcile two tables both ways", "Data quality", sql=SQL_Q6.strip()))
display(out.head())


## Results for this notebook

`Same SQL?` tells you whether both engines ran the *identical* SQL string. Where it says no, the two dialects genuinely differ and the case is written twice.

`Same answer?` is the check that matters: a fast wrong answer is worth nothing.

In [ ]:
report.headline(bench.table())
print()
display(report.results_table(bench.table()))
report.times_chart(bench.table())
bench.save()
engines.stop_spark()
